In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score, classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression, LassoCV

RANDOM_STATE = 42
DATA_PATH = Path("data/medical_insurance.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 60)

## Research questions

1. What distinct clusters emerge among insured individuals based on demographic, behavioural, and clinical variables?
2. How accurately do classification models predict insurance risk levels based on demographic, behavioural, and clinical variables?
3. Which demographic, behavioural and clinical factors exert the strongest statistical impact on annual premium?

The notebook uses clustering for RQ1, classification for RQ2, and regression for RQ3.

## 1. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Place the extracted Kaggle CSV at {DATA_PATH.resolve()} "
        "or update DATA_PATH to the location of your CSV."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.info())

## 2. Data quality assessment

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing)

display(df.describe(include="all").T)

## 3. Data preparation and feature groups

In [ ]:
# Remove exact duplicate records.
df = df.drop_duplicates().copy()

# Research predictors: demographic, behavioural and clinical variables.
demographic_features = [
    "age", "sex", "region", "urban_rural", "income", "education",
    "marital_status", "employment_status", "household_size", "dependents"
]
behavioural_features = [
    "smoker", "alcohol_freq", "visits_last_year"
]
clinical_features = [
    "bmi", "hospitalizations_last_3yrs", "days_hospitalized_last_3yrs",
    "medication_count", "systolic_bp", "diastolic_bp", "ldl", "hba1c",
    "chronic_count", "hypertension", "diabetes", "asthma", "copd",
    "cardiovascular_disease", "cancer_history", "kidney_disease",
    "liver_disease", "arthritis", "mental_health"
]

candidate_features = demographic_features + behavioural_features + clinical_features
features = [c for c in candidate_features if c in df.columns]

print("Available demographic features:", [c for c in demographic_features if c in df.columns])
print("Available behavioural features:", [c for c in behavioural_features if c in df.columns])
print("Available clinical features:", [c for c in clinical_features if c in df.columns])
print("Total predictors:", len(features))

## RQ1 – Clustering insured individuals

**Research question:** What distinct clusters emerge among insured individuals based on demographic, behavioural, and clinical variables?

K-means and DBSCAN are applied to the same prepared feature set. The person identifier and insurance outcomes (including risk score, annual premium, claims and medical costs) are excluded from clustering so that groups are formed from the specified demographic, behavioural and clinical characteristics.

### 4. Prepare data for clustering

In [ ]:
X_cluster = df[features].copy()

numeric_cluster = X_cluster.select_dtypes(include=np.number).columns.tolist()
categorical_cluster = X_cluster.select_dtypes(exclude=np.number).columns.tolist()

cluster_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cluster_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

cluster_preprocessor = ColumnTransformer([
    ("numeric", cluster_numeric_pipeline, numeric_cluster),
    ("categorical", cluster_categorical_pipeline, categorical_cluster)
])

X_cluster_ready = cluster_preprocessor.fit_transform(X_cluster)
if hasattr(X_cluster_ready, "toarray"):
    X_cluster_ready = X_cluster_ready.toarray()

print("Prepared clustering matrix:", X_cluster_ready.shape)

### 5. K-means clustering

In [ ]:
# Compare a practical range of cluster counts using silhouette score.
kmeans_scores = []
for k in range(2, 8):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_cluster_ready)
    score = silhouette_score(
        X_cluster_ready, labels,
        sample_size=min(10000, len(labels)),
        random_state=RANDOM_STATE
    )
    kmeans_scores.append({"k": k, "silhouette_score": score})

kmeans_scores_df = pd.DataFrame(kmeans_scores)
display(kmeans_scores_df)

best_k = int(kmeans_scores_df.loc[kmeans_scores_df["silhouette_score"].idxmax(), "k"])
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df["kmeans_cluster"] = kmeans.fit_predict(X_cluster_ready)

display(df["kmeans_cluster"].value_counts().sort_index().to_frame("individuals"))

### 6. DBSCAN clustering

In [ ]:
# DBSCAN can be computationally expensive on large datasets.
# Run it on a reproducible sample, then interpret the result as a sample-based analysis.
DBSCAN_SAMPLE_SIZE = min(10000, len(df))
rng = np.random.RandomState(RANDOM_STATE)
dbscan_indices = rng.choice(len(df), size=DBSCAN_SAMPLE_SIZE, replace=False)
X_dbscan_sample = X_cluster_ready[dbscan_indices]

dbscan = DBSCAN(eps=2.0, min_samples=20)
dbscan_labels = dbscan.fit_predict(X_dbscan_sample)

dbscan_summary = pd.Series(dbscan_labels, name="dbscan_cluster").value_counts().sort_index()
display(dbscan_summary.to_frame("sample_individuals"))

non_noise = dbscan_labels != -1
n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
if n_dbscan_clusters >= 2 and non_noise.sum() > n_dbscan_clusters:
    print("DBSCAN silhouette (excluding noise):",
          round(silhouette_score(X_dbscan_sample[non_noise], dbscan_labels[non_noise],
                                 sample_size=min(5000, non_noise.sum()),
                                 random_state=RANDOM_STATE), 3))
else:
    print("DBSCAN did not identify at least two non-noise clusters; silhouette is not applicable.")
print(f"DBSCAN was run on a sample of {DBSCAN_SAMPLE_SIZE:,} records.")

### 7. Interpret cluster profiles

In [ ]:
# Review the average numerical characteristics of each K-means cluster.
numeric_profile_cols = [c for c in numeric_cluster if c in df.columns]
if numeric_profile_cols:
    display(df.groupby("kmeans_cluster")[numeric_profile_cols].mean().round(2))

# Review the most common categories in each cluster.
for col in categorical_cluster:
    if col in df.columns:
        print(f"\nMost common {col} by K-means cluster:")
        display(pd.crosstab(df["kmeans_cluster"], df[col], normalize="index").round(2))

## RQ2 – Predicting insurance risk levels

**Research question:** How accurately do classification models predict insurance risk levels based on demographic, behavioural, and clinical variables?

The dataset's `is_high_risk` field is used as the classification target. `risk_score` is excluded because it may directly underpin or encode the risk label. Identifiers, premiums, claims and medical-cost outcomes are also excluded from predictors.

### 8. Prepare classification data

In [ ]:
CLASS_TARGET = "is_high_risk"
if CLASS_TARGET not in df.columns:
    raise KeyError(f"Classification target '{CLASS_TARGET}' is missing.")

X_class = df[features].copy()
y_class = df[CLASS_TARGET].copy()

# Convert common boolean/string labels into consistent categories if necessary.
if y_class.dtype == "object":
    y_class = y_class.astype(str).str.strip()

print("Risk-level distribution:")
display(y_class.value_counts(dropna=False).to_frame("count"))
display((y_class.value_counts(normalize=True) * 100).round(2).to_frame("percent"))

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y_class
)

### 9. Preprocess and train KNN and SVM

In [ ]:
num_class = Xc_train.select_dtypes(include=np.number).columns.tolist()
cat_class = Xc_train.select_dtypes(exclude=np.number).columns.tolist()

class_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_class),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_class)
])

classifiers = {
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine": SVC(kernel="rbf", class_weight="balanced")
}

classification_models = {}
classification_results = []

for name, estimator in classifiers.items():
    pipe = Pipeline([
        ("preprocessor", class_preprocessor),
        ("classifier", estimator)
    ])
    pipe.fit(Xc_train, yc_train)
    pred = pipe.predict(Xc_test)
    classification_models[name] = pipe

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(yc_test, pred),
        "Precision_weighted": precision_recall_fscore_support(
            yc_test, pred, average="weighted", zero_division=0
        )[0],
        "Recall_weighted": precision_recall_fscore_support(
            yc_test, pred, average="weighted", zero_division=0
        )[1],
        "F1_weighted": precision_recall_fscore_support(
            yc_test, pred, average="weighted", zero_division=0
        )[2]
    })

classification_results_df = pd.DataFrame(classification_results)
display(classification_results_df.round(3))

### 10. Classification reports and confusion matrices

In [ ]:
for name, pipe in classification_models.items():
    pred = pipe.predict(Xc_test)
    print(f"\n{name}")
    print(classification_report(yc_test, pred, zero_division=0))
    cm = confusion_matrix(yc_test, pred, labels=pipe.classes_)
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual {c}" for c in pipe.classes_],
        columns=[f"Predicted {c}" for c in pipe.classes_]
    )
    display(cm_df)

## RQ3 – Factors associated with annual premium

**Research question:** Which demographic, behavioural and clinical factors exert the strongest statistical impact on annual premium?

Annual premium is the outcome variable. Linear Regression and Lasso Regression are used to examine relationships between the specified predictor groups and annual premium. Coefficients indicate associations conditional on the model; they do not establish causation.

### 11. Prepare premium regression data

In [ ]:
REG_TARGET = "annual_premium"
if REG_TARGET not in df.columns:
    raise KeyError(f"Regression target '{REG_TARGET}' is missing.")

X_reg = df[features].copy()
y_reg = pd.to_numeric(df[REG_TARGET], errors="coerce")

valid_rows = y_reg.notna()
X_reg = X_reg.loc[valid_rows]
y_reg = y_reg.loc[valid_rows]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

num_reg = Xr_train.select_dtypes(include=np.number).columns.tolist()
cat_reg = Xr_train.select_dtypes(exclude=np.number).columns.tolist()

reg_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_reg),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_reg)
])

### 12. Train Linear Regression and Lasso

In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": LassoCV(
        cv=5, random_state=RANDOM_STATE, max_iter=10000, n_jobs=-1
    )
}

regression_models = {}
regression_results = []

for name, estimator in regressors.items():
    pipe = Pipeline([
        ("preprocessor", reg_preprocessor),
        ("regressor", estimator)
    ])
    pipe.fit(Xr_train, yr_train)
    pred = pipe.predict(Xr_test)
    regression_models[name] = pipe

    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(yr_test, pred),
        "RMSE": np.sqrt(mean_squared_error(yr_test, pred)),
        "R2": r2_score(yr_test, pred)
    })

regression_results_df = pd.DataFrame(regression_results)
display(regression_results_df.round(3))

### 13. Examine regression coefficients

In [ ]:
# Extract coefficients from the fitted models. Because numeric variables are
# standardised and categories are one-hot encoded, coefficient magnitudes
# can be compared cautiously within this modelling setup.
for name, pipe in regression_models.items():
    pre = pipe.named_steps["preprocessor"]
    model = pipe.named_steps["regressor"]
    feature_names = pre.get_feature_names_out()
    coef = pd.DataFrame({
        "feature": feature_names,
        "coefficient": model.coef_
    })
    coef["absolute_coefficient"] = coef["coefficient"].abs()
    coef = coef.sort_values("absolute_coefficient", ascending=False)

    print(f"\n{name}: strongest coefficients")
    display(coef.head(20).round(3))

### 14. Compare actual and predicted annual premiums

In [ ]:
best_reg_name = regression_results_df.sort_values("RMSE").iloc[0]["Model"]
best_reg_model = regression_models[best_reg_name]
premium_predictions = best_reg_model.predict(Xr_test)

plt.figure(figsize=(7, 6))
plt.scatter(yr_test, premium_predictions, alpha=0.25)
low = min(yr_test.min(), premium_predictions.min())
high = max(yr_test.max(), premium_predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual annual premium")
plt.ylabel("Predicted annual premium")
plt.title(f"Actual vs Predicted Annual Premium – {best_reg_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Export key tables for the GitHub project.
kmeans_scores_df.to_csv(OUTPUT_DIR / "kmeans_silhouette_scores.csv", index=False)
classification_results_df.to_csv(OUTPUT_DIR / "classification_model_comparison.csv", index=False)
regression_results_df.to_csv(OUTPUT_DIR / "premium_regression_model_comparison.csv", index=False)

print("Analysis tables saved to:", OUTPUT_DIR.resolve())